# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [ ]:
#Run for google Colab setup

'''
!pip install gsplat
!pip install plyfile

!git clone https://github.com/facebookresearch/uco3d.git
!pip install -e /content/uco3d/

!pip uninstall -y uco3d
import sys, importlib

# Remove ALL uco3d-related path entries (including any stale ones)
sys.path = [p for p in sys.path if 'uco3d' not in p]

# Clear cached modules
for mod in list(sys.modules):
    if mod == 'uco3d' or mod.startswith('uco3d.'):
        del sys.modules[mod]

# Add the OUTER folder so 'uco3d' resolves to /content/uco3d/uco3d as a regular package
sys.path.insert(0, '/content/uco3d')

import uco3d
print(uco3d.__file__)   # should be /content/uco3d/uco3d/__init__.py
print(uco3d.__path__)

!mkdir data/
!python uco3d/dataset_download/download_dataset.py --download_small_subset --download_folder ./data
'''


In [ ]:
import torch
import torch.nn as nn
from uco3d.dataset_utils.data_types import GaussianSplats

class POC_Network(nn.Module):
    def __init__(self, dim, bottleneck_size):
        super(POC_Network, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size),
            nn.ReLU(),
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, dim)
        )

    def forward(self, x):
        return self.model(x)
    
    def reconstruct_splats(self, x):

        reconstructed_flat = self.model(x)
        means_recon = reconstructed_flat[:, 0:3]
        sh0_recon = reconstructed_flat[:, 3:6]
        scales_recon = reconstructed_flat[:, 6:9]
        quats_recon = reconstructed_flat[:, 9:13]
        opacities_recon = reconstructed_flat[:, 13:14].squeeze(-1)

        return GaussianSplats(
            means=means_recon,
            sh0=sh0_recon,
            scales=scales_recon,
            quats=quats_recon,
            opacities=opacities_recon,
            shN=splats.shN,
            fg_mask=splats.fg_mask,
        )


In [ ]:
EPOCHS = 1000
LR = 1e-3
BOTTLENECK_SIZES = [10, 8]
SAMPLES_LIST = [1]


In [3]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

In [ ]:
from gsplat import rasterization
from uco3d.dataset_utils.utils import opencv_cameras_projection_from_uco3d
import matplotlib.pyplot as plt
from IPython.display import clear_output

def get_camera_params(frame, device):
    h, w = frame.image_rgb.shape[-2:]
    image_size_tensor = torch.tensor([[h, w]], dtype=torch.float32)

    Rcv, tvec, camera_matrix = opencv_cameras_projection_from_uco3d(
        frame.camera, image_size=image_size_tensor,
    )

    viewmats = torch.eye(4)[None].to(device)
    viewmats[:, :3, :4] = torch.cat([Rcv, tvec[..., None]], dim=-1)

    return viewmats, camera_matrix.to(device), h, w

def get_camera_params_from_camera(camera, device):
    """Convert our Cameras dataclass to gsplat viewmats + K (same math as opencv_cameras_projection_from_uco3d)."""
    h, w = int(camera.image_size[0, 0].item()), int(camera.image_size[0, 1].item())

    R_p3d = camera.R.clone()
    T_p3d = camera.T.clone()
    T_p3d[:, :2] *= -1
    R_p3d[:, :, :2] *= -1
    tvec = T_p3d
    R_cv = R_p3d.permute(0, 2, 1)

    viewmats = torch.eye(4, device=device)[None]
    viewmats[:, :3, :3] = R_cv.to(device)
    viewmats[:, :3, 3] = tvec.to(device)

    image_size_wh = torch.tensor([[w, h]], dtype=torch.float32)
    scale = image_size_wh.min(dim=1, keepdim=True)[0] / 2.0
    c0 = image_size_wh / 2.0
    focal_length = camera.focal_length * scale
    principal_point = -camera.principal_point * scale + c0

    Ks = torch.zeros(1, 3, 3, device=device)
    Ks[0, 0, 0] = focal_length[0, 0]
    Ks[0, 1, 1] = focal_length[0, 1]
    Ks[0, 0, 2] = principal_point[0, 0]
    Ks[0, 1, 2] = principal_point[0, 1]
    Ks[0, 2, 2] = 1.0

    return viewmats, Ks, h, w

def denormalize_splats(recon_tensor, raw_splats, global_stats, device):
    """Invert _normalize_with_global_stats to recover world-space splat parameters.

    raw_splats is the original (non-normalized) GaussianSplats from the batch;
    it is used to recompute the per-sample centroid and position_scale that were
    applied during normalization.
    """
    centroid = raw_splats.means.mean(dim=0, keepdim=True)          # (1, 3)
    means_centered = raw_splats.means - centroid
    position_scale = means_centered.norm(dim=1).max().clamp(min=1e-8)  # scalar

    # Tensor layout matches _normalize_with_global_stats output:
    # [means(0:3), sh0(3:6), opacities(6:7), scales(7:10), quats(10:14)]
    means    = recon_tensor[:, 0:3]  * position_scale.to(device) + centroid.to(device)
    sh0      = recon_tensor[:, 3:6]  * global_stats["sh0_std"].to(device)   + global_stats["sh0_mean"].to(device)
    opacities= (recon_tensor[:, 6:7] * global_stats["op_std"].to(device)    + global_stats["op_mean"].to(device)).squeeze(-1)
    scales   = recon_tensor[:, 7:10] * global_stats["scale_std"].to(device) + global_stats["scale_mean"].to(device)
    quats    = recon_tensor[:, 10:14]  # render_view normalises these

    return GaussianSplats(means=means, sh0=sh0, opacities=opacities, scales=scales, quats=quats, shN=None, fg_mask=None)

def render_view(recon_splats, viewmats, Ks, width, height):
    imgs, _, _ = rasterization(
        means       = recon_splats.means,
        quats       = torch.nn.functional.normalize(recon_splats.quats, p=2, dim=-1),
        scales      = torch.exp(recon_splats.scales),
        opacities   = torch.sigmoid(recon_splats.opacities.flatten()),
        colors      = recon_splats.sh0.unsqueeze(1),
        viewmats    = viewmats,
        Ks          = Ks,
        width       = width,
        height      = height,
        sh_degree   = 0,
        near_plane  = 0.01,
        backgrounds = torch.tensor([0.0, 0.0, 0.0], device=DEVICE),
        rasterize_mode = "classic",
    )
    return imgs.permute(0, 3, 1, 2)  # (1, 3, H, W)

In [ ]:
import wandb
import torch.nn.functional as F
from torch.optim import AdamW
from dataset import get_dataloader

def train_run(bottleneck_size, num_samples):
    network = POC_Network(dim=14, bottleneck_size=bottleneck_size).to(DEVICE)
    loader = get_dataloader("./data/uco3d_data_raw", batch_size=1, num_samples=num_samples)
    optim = AdamW(network.parameters(), lr=LR)

    # global_stats live on the UCO3DDataset wrapped by the Subset
    global_stats = loader.dataset.dataset.global_stats

    run = wandb.init(
        entity="leonardkuech-technical-university-of-munich",
        project="feed-forward radiance field conversion",
        config={
            "learning_rate": LR,
            "bottleneck_size": bottleneck_size,
            "dataset": "uco3d",
            "epochs": EPOCHS,
            "samples": num_samples,
            "device": str(DEVICE),
        },
        reinit=True,
    )

    for epoch in range(EPOCHS):
        optim.zero_grad()
        batch = next(iter(loader))
        x = batch[0]["splat_tensor"].to(DEVICE)
        output = network(x)
        loss = F.mse_loss(x, output)
        loss.backward()
        optim.step()

        if epoch == EPOCHS - 1:
            network.eval()
            images_to_log = []
            with torch.no_grad():
                for i in range(num_samples):
                    sample = next(iter(loader))[0]
                    x = sample["splat_tensor"].to(DEVICE)
                    recon_tensor = network(x)  # (N, 14) normalized

                    raw = sample["sequence_gaussian_splats"]

                    recon_splats = denormalize_splats(recon_tensor, raw, global_stats, DEVICE)

                    gt_splats = GaussianSplats(
                        means=raw.means.to(DEVICE),
                        sh0=raw.sh0.to(DEVICE),
                        opacities=raw.opacities.to(DEVICE),
                        scales=raw.scales.to(DEVICE),
                        quats=raw.quats.to(DEVICE),
                        shN=None,
                        fg_mask=None,
                    )

                    viewmats, Ks, h, w = get_camera_params_from_camera(sample["camera"], DEVICE)

                    gt_render    = render_view(gt_splats, viewmats, Ks, w, h)
                    recon_render = render_view(recon_splats, viewmats, Ks, w, h)

                    gt_img_np    = sample["image_rgb"].permute(1, 2, 0).clamp(0, 1).numpy()
                    gt_render_np = gt_render[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()
                    recon_np     = recon_render[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()

                    images_to_log.extend([
                        wandb.Image(gt_img_np,    caption=f"GT image (sample {i})"),
                        wandb.Image(gt_render_np, caption=f"GT splats rendered (sample {i})"),
                        wandb.Image(recon_np,     caption=f"Recon splats rendered (sample {i})"),
                    ])

            run.log({"renders": images_to_log})

        run.log({"loss": loss.item()})

    run.finish()

In [5]:
for bottleneck_size in BOTTLENECK_SIZES:
    for num_samples in SAMPLES_LIST:
        print(f"Training: bottleneck={bottleneck_size}, samples={num_samples}")
        train_run(bottleneck_size, num_samples)


Training: bottleneck=10, samples=1


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/leonardkuch/.netrc.
wandb: Currently logged in as: leonardkuech (leonardkuech-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


loss,██▇▆▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,0.00674


Training: bottleneck=10, samples=2


...

Training: bottleneck=8, samples=5


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


loss,█▅▄▆▃▃▃▄▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,0.03447


Training: bottleneck=8, samples=10


KeyboardInterrupt: 